# Cost-Aware Simulation Based Inference

**References:**
- Bharti et al. (2025) — [Cost-aware simulation-based inference](https://arxiv.org/abs/2410.07930)

Generating samples for simulation-based inference can have a high computational cost. The cost of each individual simulation often depends on on the parameter value used for simulation. Cost-aware SBI uses importance sampling to encourage sampling from the computationally cheaper parameterisations of the model. In this way it can significantly reduce simulation costs, without any changes to the simulator itself. 

## 1. Imports

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import pickle
import time
import numpy as np

import matplotlib.pyplot as plt
import bayesflow as bf

from cost_interp_model import CostInterpModel
from cost_plots import *


## 2. Generate Synthetic Cost Data

In a real scenario, cost can be any value that is meaningful for the simulator, for example wall-clock time of the simulation. In this toy example, we use a function of the SIR parameters ($\beta ,\gamma$) to mimic the cost behaviour of the model. This can be any model which 

In [ ]:
def make_synthetic_cost_data(cost_data_dir="cost_data_sir"):
    # Initialize the base SIR simulator
    sir_sim = bf.simulators.benchmark_simulators.SIR()

    os.makedirs(cost_data_dir, exist_ok=True)

    print("Generating synthetic cost data...")
    num_cost_samples = 20
    for i in range(num_cost_samples):
        theta = sir_sim.prior()
        # Simulate cost: higher beta -> higher cost
        actual_cost = np.exp(-theta[0])
        #3 - np.abs(theta[1]) + np.random.normal(0, 0.1)

        with open(f"{cost_data_dir}/sample_{i}.pkl", "wb") as f:
            pickle.dump({"theta": theta, "cost": actual_cost}, f)
            
    return sir_sim


cost_data_dir = "cost_data_sir"
sir_sim = make_synthetic_cost_data(cost_data_dir)


## 3. Fit Cost Interpolation Model

We use a small test Gaussian Process (GP) called `CostInterpModel` to predict the cost given the parameters $\theta$.
This is a user input so can be very varied. 

In [ ]:
print("Fitting cost interpolation model...")
cost_model = CostInterpModel(root=cost_data_dir).fit(length_scale=3.0)


We can visualise how the cost changes with the parameters $\beta$ and $\gamma$. 

In [ ]:
fig = plot_cost_landscape(cost_model)
plt.show(fig)

## 4. Initialize Cost-Aware Simulator

The `CostAwareSimulator` wraps the base simulator and uses the fitted cost model to filter samples based on a regularised cost function $g(c(\theta))$.

In [ ]:
cost_aware_sim = bf.simulators.CostAwareSimulator(simulator=sir_sim, cost_model=cost_model)


## 5. Cost-Aware Sampling

We use rejection sampling to obtain a set of cost-efficient parameters. For constructing the sampler, we use a regularisation of the cost                                
                                                                                                                                                
   \begin{equation}                                                                                                                             
       g(c) = \max\left(g_{\min}, (c + g_{\min})^k\right)                                                                                       
   \end{equation}      
                                                                            
with $g_{\min}$ is a minimum value (offset) to prevent the weight from becoming zero or too small and $k$ is a power factor for cost regularization. Both parameters are configurable through the class properties of the CostAwareSampler.
                                                   
Candidates $\theta$ drawn from the prior distribution are accepted based on a regularization of the cost $g(c)$. The acceptance probability is 
\begin{equation}                                                                                                                             
       P(\text{accept} \mid \theta) = \frac{g_{\min}}{g(c(\theta))}                                                                             
   \end{equation}   

In [ ]:
print("\nTesting cost-aware sampling...")
num_samples = 1000

#Use the cost-aware sampling method
accepted_samples = cost_aware_sim.sample(batch_shape=(num_samples,1))
accepted_theta = accepted_samples.get("parameters")

#may return more than the requested number of samples. This is because of the way that rejection_sample works, in batches until it reaches the total
#number of samples
print(f"Successfully sampled {len(accepted_theta)} cost-efficient parameters.")


The samples generated by cost-aware sampling can be compared with those taken directly from the prior. 

In [ ]:

plot_prior_samples(cost_model,sir_sim,accepted_theta)

In [ ]:
plot_histograms(sir_sim, accepted_theta)

## 7. Performance Metrics

We evaluate the effectiveness of the cost-aware sampling using Effective Sample Size (ESS) and Computational Gain (CG) for the accepted samples.

                                                                   
   \begin{equation}                                                                                                                             
       \text{ESS} = \frac{\left( \sum_{i=1}^{N} g(c(\theta_i)) \right)^2}{N \sum_{i=1}^{N} g(c(\theta_i))^2}                                                          
   \end{equation}         


In [50]:
metrics = cost_aware_sim.compute_metrics({"theta": accepted_theta})
print(f"  Effective Sample Size: {metrics['ess']:.2f}")

  Effective Sample Size: 0.97


Computational gain is a measure of the reduction of the cost, given by the cost function, by using the cost-aware samples over plainly sampling the prior. The metric is calculated as the ratio of the average cost of samples drawn from the original prior to the average cost of the samples accepted by the cost-aware sampler.


\begin{equation*}                 
\text{CG} = \frac{\mathbb{E}_{\theta \sim \pi(\theta)} (c(\theta))}{\mathbb{E}_{\theta \sim \pi_{\text{aware}}(\theta)} (c(\theta))}    
\end{equation*}                                        
       
This is estimated as                 
\begin{equation*}                                              \text{CG} \approx \frac{\frac{1}{N} \sum_{j=1}^{N} c(\theta_j^{\text{prior}})}{\frac{1}{N} \sum_{i=1}^{N} c(\theta_i^{\text{accepted}})}
\end{equation*}     
where samples $\theta_j^{\text{prior}}$ are taken from the original prior distribution. 


In [ ]:
print(f"  Computational Gain:  {metrics['cg']:.2f}")

## 9. Computing Training Weights

Finally, we sample the expensive simulator using the cost aware samples. Once we have the samples, we compute their relative weights 

To not waste data, we add the initial samples from the cost model fit to the training batch. We then compute training weights over the whole batch. 

  We use self-normalised weights $w_i$ for the i-th accepted sample $\theta_i$. Let $N$ be the total number of accepted samples in the batch. 
\begin{equation}                                                                                                                             
       w_i = \frac{g(c(\theta_i))}{\sum_{j=1}^{N} g(c(\theta_j))}                                                                               
   \end{equation}                                                                

In [ ]:
num_accepted = len(accepted_theta)

if num_accepted > 0:
    
    train_observations = [sir_sim.observation_model(t) for t in accepted_theta]
    print(f"\nSuccessfully simulated {len(train_observations)} samples using the expensive simulator.")

    # Load and include dummy data from cost_data_dir
    dummy_files = [f for f in os.listdir(cost_data_dir) if f.endswith('.pkl')]
    for f_name in dummy_files:
        with open(os.path.join(cost_data_dir, f_name), 'rb') as f:
            data = pickle.load(f)
            # In this dummy case, we use the theta as a proxy for observation
            # or we could run the simulator on it
            train_observations.append(sir_sim.observation_model(data['theta']))
            
    train_observations = np.array(train_observations)

    # Compute weights for all samples (accepted + dummy)
    # First, get the parameters for dummy data
    dummy_thetas = []
    dummy_files = [f for f in os.listdir(cost_data_dir) if f.endswith('.pkl')]
    for f_name in dummy_files:
        with open(os.path.join(cost_data_dir, f_name), 'rb') as f:
            dummy_thetas.append(pickle.load(f)['theta'])
    
    all_theta = np.concatenate([accepted_theta, np.array(dummy_thetas)], axis=0)

    train_weights = cost_aware_sim.compute_weights(all_theta)



We can then plug into the existing Bayesflow framework to complete inference. The training weights are passed to the 


In [ ]:
train_weights = np.asarray(train_weights, dtype=np.float32).reshape(-1, 1)


train_data = {
    "theta": np.asarray(all_theta),
    "observations": np.asarray(train_observations),
    "weights": train_weights}


adapter = (
    bf.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .concatenate(["theta"], into="inference_variables")
    .concatenate(["observations"], into="inference_conditions")
    .rename("weights", "sample_weight")
)

inference_net = bf.networks.PointNetwork(points="mean")

workflow = bf.BasicWorkflow(
    simulator=cost_aware_sim,
    adapter=adapter,
    inference_network=inference_net,
)

In [52]:
history = workflow.fit_offline(                                                                                                                               
       data=train_data,                                                                                                                                
       epochs=20,                                                                                                                                     
       batch_size=32                                                                                                                                   
   )

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.


Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - PointNetwork/mean: 2.4255e-04 - loss: 2.4255e-04
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - PointNetwork/mean: 2.3758e-04 - loss: 2.3758e-04
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - PointNetwork/mean: 2.3169e-04 - loss: 2.3169e-04
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - PointNetwork/mean: 2.2660e-04 - loss: 2.2660e-04
Epoch 5/20
25/40 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - PointNetwork/mean: 1.9961e-05 - loss: 1.9961e-05

KeyboardInterrupt: 

In [ ]:
#plot some diagnostics.                                                                                                                                                     
def get_data(workflow, test_dataset, parameter_name):                                                                                               
    """Helper to extract truths, estimates, and costs."""                                                                                           
    posteriors = workflow.approximator.estimate(test_dataset)                                                                                       
    truths = np.array([sample[parameter_name] for sample in test_dataset.truths])                                                                   
    estimates = np.array([posteriors[i][parameter_name] for i in range(len(posteriors))])                                                           
    costs = np.array([sample['cost'] for sample in test_dataset.observations])                                                                      
    return truths, estimates, costs                                                                                                                 
                                                                                                                                                    
def plot_accuracy_vs_cost(workflow, test_dataset, parameter_name):                                                                                  
    """Plot 1: The Pareto Frontier (RMSE vs Simulation Cost)."""                                                                                    
    truths, estimates, costs = get_data(workflow, test_dataset, parameter_name)                                                                     
                                                                                                                                                    
    # Sort by cost                                                                                                                                  
    idx = np.argsort(costs)                                                                                                                         
    sorted_costs = costs[idx]                                                                                                                       
    sorted_truths = truths[idx]                                                                                                                     
    sorted_estimates = estimates[idx]                                                                                                               
                                                                                                                                                    
    # Create cost buckets to evaluate RMSE                                                                                                          
    cost_bins = np.linspace(sorted_costs.min(), sorted_costs.max(), 20)                                                                             
    rmse_values = []                                                                                                                                
    metric = bf.diagnostics.metrics.root_mean_squard_error.RootMeanSquaredError()                                                                                                                 
                                                                                                                                                    
    for bin_edge in cost_bins:                                                                                                                      
        mask = sorted_costs <= bin_edge                                                                                                             
        rmse_values.append(metric(sorted_truths[mask], sorted_estimates[mask]))                                                                     
                                                                                                                                                    
    plt.figure(figsize=(8, 5))                                                                                                                      
    plt.plot(cost_bins, rmse_values, marker='o', linestyle='-', color='#1f77b4')                                                                    
    plt.title(f"Accuracy vs. Cost ({parameter_name})")                                                                                              
    plt.xlabel("Max Simulation Cost Allowed")                                                                                                       
    plt.ylabel("RMSE")                                                                                                                              
    plt.grid(True, alpha=0.3)                                                                                                                       
    plt.show()                                                                                                                                      
                                                                                                                                                    
def plot_cost_distribution(test_dataset):                                                                                                           
    """Plot 2: Distribution of Simulation Costs."""                                                                                                 
    costs = np.array([sample['cost'] for sample in test_dataset.observations])                                                                      
                                                                                                                                                    
    plt.figure(figsize=(8, 5))                                                                                                                      
    plt.hist(costs, bins=30, color='#2ca02c', edgecolor='white', alpha=0.8)                                                                         
    plt.title("Distribution of Simulation Costs")                                                                                                   
    plt.xlabel("Cost")                                                                                                                              
    plt.ylabel("Number of Samples")                                                                                                                 
    plt.grid(axis='y', alpha=0.3)                                                                                                                   
    plt.show()                                                                                                                                      
                                                                                                                                                    
def plot_recovery_by_cost(workflow, test_dataset, parameter_name):                                                                                  
    """Plot 3: Parameter Recovery colored by cost."""                                                                                               
    truths, estimates, costs = get_data(workflow, test_dataset, parameter_name)                                                                     
                                                                                                                                                    
    plt.figure(figsize=(8, 6))                                                                                                                      
    sc = plt.scatter(truths, estimates, c=costs, cmap='viridis', alpha=0.6, edgecolors='none')                                                      
    plt.plot([truths.min(), truths.max()], [truths.min(), truths.max()], 'r--', lw=2)                                                               
                                                                                                                                                    
    plt.colorbar(sc, label='Simulation Cost')                                                                                                       
    plt.title(f"Recovery colored by Cost ({parameter_name})")                                                                                       
    plt.xlabel("True Value")                                                                                                                        
    plt.ylabel("Estimated Value")                                                                                                                   
    plt.grid(True, alpha=0.3)                                                                                                                       
    plt.show()                        

In [ ]:
plot_accuracy_vs_cost(workflow, train_data, "gamma")

## 10. Multiple Importance sampling 
When the true posterior lies in the computationally costly region, choosing a single penalty function based on CG and ESS alone may lead to sub-optimal results. This is why we may want to consider multiple cost-aware priors. 

This is done by first sampling 


In [ ]:
regularisation_powers = [0,1,2,3]
num_thetas = 1000 #total number of samples needed


for k in regularisation_powers:

    print(k)